# P0 stage 1 full-data FP32 — CVE-only `target`

Chạy full **44.331 train / 11.089 validation** bằng **FP32**, sau diagnostic
16.000 dòng cho thấy FP16 gây NaN.

- `t5-base`, `target` full, max 512/512.
- Batch 8/GPU + grad accumulation 4 nếu fit; fallback batch 4 + grad 8.
- 5 epoch nếu batch 8; 4 epoch nếu phải fallback batch 4 (do quota 8,61h).
- LR `5e-5`, schedule 50 epoch, warmup 10%, `stop-after-epoch`.
- Lưu checkpoint từng epoch; `--abort-on-nan`.
- Eval full validation sau khi train; probe 1.000 dòng cho checkpoint epoch 1.

Đây vẫn là **partial probe**, không phải 10-epoch paper-final.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
import torch

hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")
DATA_DIR = Path(hits[0]).parent
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("DATA_DIR:", DATA_DIR)
print("train exists:", (DATA_DIR / "flow_train.jsonl").exists())
print("validation exists:", (DATA_DIR / "flow_validation.jsonl").exists())


In [ ]:
import subprocess, sys, shutil
from pathlib import Path

REPO = Path("/kaggle/working/Vuln2Action-Lab")
DATA_DIR = Path(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True)[0]).parent
SMOKE = Path("/kaggle/working/smoke_bs8")
FULL = Path("/kaggle/working/t5_stage1_paper_target_fp32")
if SMOKE.exists():
    shutil.rmtree(SMOKE, ignore_errors=True)

def train_cmd(bs, grad, epochs, output_dir, max_steps=None):
    cmd = [
        sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
        "scripts/train_t5.py",
        "--dataset", str(DATA_DIR),
        "--output-dir", str(output_dir),
        "--model-name", "t5-base",
        "--target-field", "target",
        "--max-input-length", "512",
        "--max-target-length", "512",
        "--per-device-batch-size", str(bs),
        "--gradient-accumulation-steps", str(grad),
        "--learning-rate", "5e-5",
        "--num-train-epochs", "50",
        "--save-steps", "693",
        "--eval-steps", "693",
        "--logging-steps", "50",
        "--save-total-limit", "6",
        "--no-load-best-model-at-end",
        "--no-fp16",
        "--gradient-checkpointing",
    ]
    if max_steps is not None:
        cmd += ["--max-steps", str(max_steps), "--save-steps", "1000", "--eval-steps", "1000", "--logging-steps", "1"]
    else:
        cmd += ["--stop-after-epoch", str(epochs)]
    return cmd

print("=== smoke test batch 8 fp32 ===")
smoke = subprocess.run(train_cmd(8, 4, 1, SMOKE, max_steps=5), cwd=REPO, text=True)
BS = 8
GRAD = 4
EPOCHS = 5
if smoke.returncode != 0:
    print("batch 8 smoke failed; fallback to batch 4 / grad 8 / 4 epochs")
    BS, GRAD, EPOCHS = 4, 8, 4
    if SMOKE.exists():
        shutil.rmtree(SMOKE, ignore_errors=True)
    smoke = subprocess.run(train_cmd(4, 8, 1, SMOKE, max_steps=5), cwd=REPO, text=True)
    if smoke.returncode != 0:
        raise RuntimeError("both batch 8 and batch 4 smoke tests failed")
shutil.rmtree(SMOKE, ignore_errors=True)
torch.cuda.empty_cache()
print(f"selected: bs={BS} grad={GRAD} epochs={EPOCHS}")


In [ ]:
print("=== full training ===")
cmd = train_cmd(BS, GRAD, EPOCHS, FULL)
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


In [ ]:
from pathlib import Path
import subprocess, sys

print("=== eval epoch-1 checkpoint on 1.000 rows ===")
subprocess.run([
    sys.executable, "scripts/evaluate_t5.py",
    "--checkpoint", str(FULL / "checkpoint-693"),
    "--dataset", str(DATA_DIR),
    "--split", "validation",
    "--target-field", "target",
    "--output-dir", "/kaggle/working/t5_stage1_fp32_eval_epoch1_1000",
    "--max-input-length", "512",
    "--max-target-length", "512",
    "--batch-size", "4",
    "--num-beams", "4",
    "--limit", "1000",
    "--no-fp16",
], cwd=REPO, check=True)

print("=== eval final checkpoint on full 11.089 rows ===")
subprocess.run([
    sys.executable, "scripts/evaluate_t5.py",
    "--checkpoint", str(FULL / "final"),
    "--dataset", str(DATA_DIR),
    "--split", "validation",
    "--target-field", "target",
    "--output-dir", "/kaggle/working/t5_stage1_fp32_eval_full",
    "--max-input-length", "512",
    "--max-target-length", "512",
    "--batch-size", "4",
    "--num-beams", "4",
    "--no-fp16",
], cwd=REPO, check=True)


In [ ]:
import json, math
from pathlib import Path

print("=== training config ===")
cfg = json.loads((FULL / "training_config.json").read_text(encoding="utf-8"))
print(json.dumps(cfg, indent=2, ensure_ascii=False))

print("\n=== trainer state ===")
states = sorted(FULL.glob("checkpoint-*/trainer_state.json"))
if (FULL / "trainer_state.json").exists():
    states.append(FULL / "trainer_state.json")
for p in states:
    state = json.loads(p.read_text(encoding="utf-8"))
    bad = []
    for row in state.get("log_history", []):
        for key in ("loss", "grad_norm", "eval_loss"):
            value = row.get(key)
            if isinstance(value, (int, float)) and not math.isfinite(value):
                bad.append((key, value, row.get("step")))
    print(p, "global_step", state.get("global_step"), "epoch", state.get("epoch"), "nonfinite", bad)
    for row in state.get("log_history", [])[-8:]:
        print(json.dumps(row, ensure_ascii=False))

for out, label in [
    (Path("/kaggle/working/t5_stage1_fp32_eval_epoch1_1000"), "epoch1_1000"),
    (Path("/kaggle/working/t5_stage1_fp32_eval_full"), "full"),
]:
    print(f"\n=== metrics {label} ===")
    m = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
    print(json.dumps(m, indent=2, ensure_ascii=False))
    rows = [json.loads(line) for line in (out / "predictions.jsonl").read_text(encoding="utf-8").splitlines()]
    empty = sum(1 for r in rows if not r.get("prediction", "").strip())
    print(f"empty predictions: {empty}/{len(rows)}")
    for i, r in enumerate(rows[:2]):
        print(f"[{i}] {r.get('cve_id')}")
        print("REF:", (r.get("reference") or "")[:250])
        print("PRED:", (r.get("prediction") or "")[:250])


## Cách đọc

- Nếu `nonfinite` rỗng, prediction không rỗng và full-validation BLEU > 0: FP32 full-data
  ổn định; đây là kết quả partial hợp lệ (số epoch ghi trong config).
- Nếu NaN: dừng, dùng checkpoint epoch trước đó; không dùng `final`.
- Không so trực tiếp BLEU partial với BLEU `0,7589` của paper; phải ghi rõ số epoch
  và việc chưa chạy đủ 10 epoch.
